#Collections Data Analyst Assignment 
#Raw Data Profiling

# Purpose:
# 1. Inventory all company-provided datasets
# 2. Inspect schemas and data quality
# 3. Identify potential keys and relationships
# 4. Detect issues before any cleaning or transformation
#
# Important:
# The files in data/raw/ are the original company-provided data.
# They must not be modified.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Raw directory exists:", RAW_DIR.exists())

Project root: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics
Raw data directory: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\raw
Raw directory exists: True


In [2]:
files = sorted(RAW_DIR.iterdir())

print(f"Total files found: {len(files)}")

for file in files:
    print(file.name)

Total files found: 19
account_status_history.csv
accounts.csv
agent_sessions.csv
agents.csv
borrowers.csv
call_attempts.csv
call_dispositions.csv
calls.csv
campaigns.csv
complaints.csv
daily_targeting.csv
data_dictionary.csv
field_visits.csv
payments.csv
promises_to_pay.csv
README.md
sms_events.csv
vendor_telephony.csv
whatsapp_events.csv


In [3]:
#DATASET INVENTORY
csv_files = sorted(RAW_DIR.glob("*.csv"))

inventory = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    inventory.append({
        "dataset": file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,dataset,rows,columns,memory_mb
0,account_status_history.csv,60000,8,27.73
1,accounts.csv,30000,11,13.79
2,agent_sessions.csv,15000,7,6.03
3,agents.csv,30000,8,13.73
4,borrowers.csv,30600,8,12.98
5,call_attempts.csv,120000,9,56.13
6,call_dispositions.csv,35000,8,16.08
7,calls.csv,91350,11,52.35
8,campaigns.csv,120,7,0.05
9,complaints.csv,8000,9,4.13


In [4]:
#SCHEMA INVENTORY
schema_rows = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    for column in df.columns:
        schema_rows.append({
            "dataset": file.name,
            "column": column,
            "dtype": str(df[column].dtype),
            "null_count": int(df[column].isna().sum()),
            "null_pct": round(df[column].isna().mean() * 100, 2),
            "unique_values": int(df[column].nunique(dropna=True))
        })

schema_df = pd.DataFrame(schema_rows)

schema_df

,dataset,column,dtype,null_count,null_pct,unique_values
0,account_status_history.csv,history_id,str,0,0.0,60000
1,account_status_history.csv,account_id,str,0,0.0,25999
2,account_status_history.csv,borrower_id,str,0,0.0,11916
3,account_status_history.csv,event_at,str,0,0.0,59898
4,account_status_history.csv,status,str,0,0.0,7
...,...,...,...,...,...,...
141,whatsapp_events.csv,event_at,str,0,0.0,59892
142,whatsapp_events.csv,message_id,str,0,0.0,34831
143,whatsapp_events.csv,event_type,str,0,0.0,6
144,whatsapp_events.csv,template_code,str,0,0.0,5


In [5]:
#MISSING DATA SUMMARY
missing_df = (
    schema_df[schema_df["null_count"] > 0]
    .sort_values(["null_pct", "dataset"], ascending=[False, True])
    .reset_index(drop=True)
)

missing_df

,dataset,column,dtype,null_count,null_pct,unique_values
0,borrowers.csv,email,str,895,2.92,15377
1,borrowers.csv,phone,float64,614,2.01,29395
2,call_attempts.csv,vendor_id,str,2400,2.00,15
3,calls.csv,agent_id,str,1827,2.00,1000
4,accounts.csv,borrower_id,str,455,1.52,10943
5,payments.csv,payment_reference,str,382,1.50,20821
6,field_visits.csv,scheduled_at,str,250,1.00,24730


In [6]:
# EXACT DUPLICATE ROWs
duplicate_rows = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    duplicate_rows.append({
        "dataset": file.name,
        "total_rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pct": round(df.duplicated().mean() * 100, 2)
    })

duplicates_df = pd.DataFrame(duplicate_rows)

duplicates_df

,dataset,total_rows,duplicate_rows,duplicate_pct
0,account_status_history.csv,60000,0,0.00
1,accounts.csv,30000,0,0.00
2,agent_sessions.csv,15000,0,0.00
3,agents.csv,30000,0,0.00
4,borrowers.csv,30600,600,1.96
5,call_attempts.csv,120000,0,0.00
6,call_dispositions.csv,35000,0,0.00
7,calls.csv,91350,1271,1.39
8,campaigns.csv,120,0,0.00
9,complaints.csv,8000,0,0.00


In [7]:
#CANDIDATE KEY ANALYSIS
key_results = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    for column in df.columns:
        # Skip columns with missing values
        if df[column].isna().any():
            continue

        unique_count = df[column].nunique()
        row_count = len(df)

        if unique_count == row_count:
            key_results.append({
                "dataset": file.name,
                "candidate_key": column,
                "rows": row_count,
                "unique_values": unique_count,
                "unique_pct": 100.0
            })

candidate_keys_df = pd.DataFrame(key_results)

candidate_keys_df

,dataset,candidate_key,rows,unique_values,unique_pct
0,account_status_history.csv,history_id,60000,60000,100.0
1,accounts.csv,account_id,30000,30000,100.0
2,agent_sessions.csv,session_id,15000,15000,100.0
3,call_attempts.csv,attempt_id,120000,120000,100.0
4,call_dispositions.csv,disposition_id,35000,35000,100.0
5,campaigns.csv,campaign_id,120,120,100.0
6,campaigns.csv,start_at,120,120,100.0
7,campaigns.csv,end_at,120,120,100.0
8,complaints.csv,complaint_id,8000,8000,100.0
9,daily_targeting.csv,target_id,45000,45000,100.0


In [8]:
# ID / FOREIGN KEY CANDIDATE INVENTORY
id_columns = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    for column in df.columns:
        if "id" in column.lower():
            id_columns.append({
                "dataset": file.name,
                "column": column,
                "dtype": str(df[column].dtype),
                "rows": len(df),
                "unique_values": df[column].nunique(dropna=True),
                "null_values": df[column].isna().sum()
            })

id_columns_df = pd.DataFrame(id_columns)

id_columns_df

,dataset,column,dtype,rows,unique_values,null_values
0,account_status_history.csv,history_id,str,60000,60000,0
1,account_status_history.csv,account_id,str,60000,25999,0
2,account_status_history.csv,borrower_id,str,60000,11916,0
3,accounts.csv,account_id,str,30000,30000,0
4,accounts.csv,borrower_id,str,30000,10943,455
5,agent_sessions.csv,session_id,str,15000,15000,0
6,agent_sessions.csv,agent_id,str,15000,1000,0
7,agent_sessions.csv,device_id,str,15000,1500,0
8,agents.csv,agent_id,str,30000,1000,0
9,agents.csv,vendor_id,str,30000,15,0


In [9]:
#DATE / TIMESTAMP COVERAGE
date_results = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    for column in df.columns:
        if any(keyword in column.lower() for keyword in [
            "date", "time", "timestamp", "at"
        ]):
            
            parsed = pd.to_datetime(df[column], errors="coerce", utc=True)

            valid = parsed.dropna()

            if len(valid) > 0:
                date_results.append({
                    "dataset": file.name,
                    "column": column,
                    "valid_dates": len(valid),
                    "min_timestamp": valid.min(),
                    "max_timestamp": valid.max(),
                    "invalid_dates": parsed.isna().sum()
                })

date_coverage_df = pd.DataFrame(date_results)

date_coverage_df

C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\3195297502.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", utc=True)
C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\3195297502.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", utc=True)
C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\3195297502.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", utc=True)
C:\Users\This PC\AppData\Local\Temp\ipykernel_

,dataset,column,valid_dates,min_timestamp,max_timestamp,invalid_dates
0,account_status_history.csv,event_at,60000,2026-01-01 00:01:08+00:00,2026-08-08 23:50:45+00:00,0
1,account_status_history.csv,recorded_at,60000,2025-12-31 01:26:29+00:00,2026-08-09 22:02:27+00:00,0
2,accounts.csv,opened_at,30000,2024-01-01 00:02:27+00:00,2025-11-30 23:52:36+00:00,0
3,agent_sessions.csv,login_at,15000,2026-01-01 00:01:57+00:00,2026-08-08 23:51:54+00:00,0
4,agent_sessions.csv,logout_at,15000,2026-01-01 04:48:48+00:00,2026-08-09 07:50:12+00:00,0
5,agents.csv,joined_at,30000,2024-01-01 00:10:05+00:00,2025-11-30 23:23:30+00:00,0
6,agents.csv,updated_at,30000,2025-01-01 00:57:32+00:00,2026-08-03 23:45:38+00:00,0
7,borrowers.csv,created_at,30600,2025-01-01 00:14:38+00:00,2026-08-03 23:37:55+00:00,0
8,borrowers.csv,updated_at,30600,2025-01-01 00:19:40+00:00,2026-08-03 23:48:36+00:00,0
9,call_attempts.csv,event_at,120000,2026-01-01 00:01:18+00:00,2026-08-08 23:58:40+00:00,0


In [10]:
#LOAD ALL COMPANY-PROVIDED DATASETS
datasets = {}

for file in sorted(RAW_DIR.glob("*.csv")):
    datasets[file.stem] = pd.read_csv(file, low_memory=False)

print(f"Datasets loaded: {len(datasets)}")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Datasets loaded: 18
account_status_history: 60,000 rows × 8 columns
accounts: 30,000 rows × 11 columns
agent_sessions: 15,000 rows × 7 columns
agents: 30,000 rows × 8 columns
borrowers: 30,600 rows × 8 columns
call_attempts: 120,000 rows × 9 columns
call_dispositions: 35,000 rows × 8 columns
calls: 91,350 rows × 11 columns
campaigns: 120 rows × 7 columns
complaints: 8,000 rows × 9 columns
daily_targeting: 45,000 rows × 7 columns
data_dictionary: 143 rows × 3 columns
field_visits: 25,000 rows × 10 columns
payments: 25,500 rows × 9 columns
promises_to_pay: 18,000 rows × 9 columns
sms_events: 45,000 rows × 8 columns
vendor_telephony: 15 rows × 6 columns
whatsapp_events: 60,600 rows × 8 columns


In [11]:
#FOREIGN KEY RELATIONSHIP CHECK
relationship_results = []

for child_name, child_df in datasets.items():

    for column in child_df.columns:

        if "id" not in column.lower():
            continue

        # Check whether the same-named ID exists in another dataset
        for parent_name, parent_df in datasets.items():

            if child_name == parent_name:
                continue

            if column in parent_df.columns:

                child_values = set(child_df[column].dropna().astype(str).unique())
                parent_values = set(parent_df[column].dropna().astype(str).unique())

                if len(child_values) > 0:
                    matched = len(child_values & parent_values)
                    match_pct = matched / len(child_values) * 100

                    relationship_results.append({
                        "child_dataset": child_name,
                        "id_column": column,
                        "possible_parent_dataset": parent_name,
                        "child_unique_ids": len(child_values),
                        "matching_parent_ids": matched,
                        "match_pct": round(match_pct, 2)
                    })

relationships_df = pd.DataFrame(relationship_results)

relationships_df.sort_values(
    ["id_column", "match_pct"],
    ascending=[True, False]
)

,child_dataset,id_column,possible_parent_dataset,child_unique_ids,matching_parent_ids,match_pct
0,account_status_history,account_id,accounts,25999,25999,100.0
71,call_attempts,account_id,accounts,29451,29451,100.0
104,call_dispositions,account_id,accounts,20603,20603,100.0
136,calls,account_id,accounts,28408,28408,100.0
171,complaints,account_id,accounts,7034,7034,100.0
...,...,...,...,...,...,...
166,calls,vendor_id,call_attempts,15,15,100.0
167,calls,vendor_id,vendor_telephony,15,15,100.0
310,vendor_telephony,vendor_id,agents,15,15,100.0
311,vendor_telephony,vendor_id,call_attempts,15,15,100.0


In [12]:
#CORE BUSINESS RELATIONSHIP CHECK

core_relationships = [
    ("accounts", "borrowers", "borrower_id"),
    ("payments", "accounts", "account_id"),
    ("calls", "accounts", "account_id"),
    ("call_attempts", "calls", "call_id"),
    ("call_dispositions", "calls", "call_id"),
    ("promises_to_pay", "accounts", "account_id"),
    ("field_visits", "accounts", "account_id"),
    ("whatsapp_events", "accounts", "account_id"),
    ("sms_events", "accounts", "account_id"),
    ("daily_targeting", "accounts", "account_id"),
    ("account_status_history", "accounts", "account_id"),
    ("agent_sessions", "agents", "agent_id"),
    ("calls", "agents", "agent_id"),
    ("calls", "vendor_telephony", "vendor_id"),
    ("campaigns", "daily_targeting", "campaign_id"),
]

core_results = []

for child, parent, key in core_relationships:

    if child not in datasets or parent not in datasets:
        continue

    child_df = datasets[child]
    parent_df = datasets[parent]

    if key not in child_df.columns or key not in parent_df.columns:
        core_results.append({
            "child": child,
            "parent": parent,
            "key": key,
            "status": "COLUMN NOT PRESENT"
        })
        continue

    child_ids = set(child_df[key].dropna().astype(str).unique())
    parent_ids = set(parent_df[key].dropna().astype(str).unique())

    unmatched = child_ids - parent_ids

    core_results.append({
        "child": child,
        "parent": parent,
        "key": key,
        "child_unique_ids": len(child_ids),
        "parent_unique_ids": len(parent_ids),
        "unmatched_child_ids": len(unmatched),
        "unmatched_pct": round(
            len(unmatched) / len(child_ids) * 100, 2
        ) if child_ids else 0,
        "status": "CHECKED"
    })

core_relationships_df = pd.DataFrame(core_results)

core_relationships_df

,child,parent,key,child_unique_ids,parent_unique_ids,unmatched_child_ids,unmatched_pct,status
0,accounts,borrowers,borrower_id,10943,11015,897,8.2,CHECKED
1,payments,accounts,account_id,16934,30000,0,0.0,CHECKED
2,calls,accounts,account_id,28408,30000,0,0.0,CHECKED
3,call_attempts,calls,call_id,66244,90000,0,0.0,CHECKED
4,call_dispositions,calls,call_id,28971,90000,0,0.0,CHECKED
5,promises_to_pay,accounts,account_id,13532,30000,0,0.0,CHECKED
6,field_visits,accounts,account_id,16908,30000,0,0.0,CHECKED
7,whatsapp_events,accounts,account_id,25924,30000,0,0.0,CHECKED
8,sms_events,accounts,account_id,23207,30000,0,0.0,CHECKED
9,daily_targeting,accounts,account_id,23344,30000,0,0.0,CHECKED


In [13]:
#PAYMENT DATA STRUCTURE
payments = datasets["payments"].copy()

print("Payment dataset shape:", payments.shape)
print("\nColumns:")
print(payments.columns.tolist())

print("\nData types:")
display(payments.dtypes.to_frame("dtype"))

print("\nFirst 5 records:")
display(payments.head())

Payment dataset shape: (25500, 9)

Columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

Data types:


,dtype
payment_id,str
account_id,str
borrower_id,str
event_at,str
payment_reference,str
amount,float64
payment_status,str
payment_method,str
provider_id,str



First 5 records:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013


In [14]:
#PAYMENT IDENTIFIER QUALITY
payment_id_columns = [
    col for col in payments.columns
    if "id" in col.lower()
    or "reference" in col.lower()
]

payment_id_summary = []

for col in payment_id_columns:
    payment_id_summary.append({
        "column": col,
        "rows": len(payments),
        "unique_values": payments[col].nunique(dropna=True),
        "null_values": payments[col].isna().sum(),
        "duplicate_values": payments[col].duplicated(keep=False).sum()
    })

payment_id_summary_df = pd.DataFrame(payment_id_summary)

payment_id_summary_df

,column,rows,unique_values,null_values,duplicate_values
0,payment_id,25500,25000,0,1000
1,account_id,25500,16934,0,14890
2,borrower_id,25500,10474,0,22455
3,payment_reference,25500,20821,382,8424
4,provider_id,25500,15,0,25500


In [15]:
#  PAYMENT STATUS AND AMOUNT PROFILE
print("Payment status distribution:")
display(
    payments["payment_status"]
    .value_counts(dropna=False)
    .rename_axis("payment_status")
    .reset_index(name="records")
)

print("\nPayment amount summary:")
display(
    payments["amount"].describe()
)

Payment status distribution:


,payment_status,records
0,SUCCESS,17880
1,FAILED,3744
2,PENDING,2592
3,REVERSED,1284



Payment amount summary:


count     25500.000000
mean      75186.612437
std       43496.079777
min         103.680000
25%       37234.580000
50%       75159.700000
75%      112925.542500
max      149993.550000
Name: amount, dtype: float64

In [16]:
# DUPLICATE PAYMENT REFERENCES
duplicate_reference_mask = (
    payments["payment_reference"]
    .notna()
    & payments["payment_reference"].duplicated(keep=False)
)

duplicate_payment_references = (
    payments.loc[duplicate_reference_mask]
    .sort_values("payment_reference")
)

print(
    "Rows belonging to repeated payment references:",
    len(duplicate_payment_references)
)

print(
    "Unique repeated payment references:",
    duplicate_payment_references["payment_reference"].nunique()
)

display(duplicate_payment_references.head(30))

Rows belonging to repeated payment references: 8042
Unique repeated payment references: 3745


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
25010,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
1387,PAYMENT0001388,ACC0019483,BRW0003324,2026-01-31 05:00:49,TXN0000000027,53610.92,FAILED,CARD,VND0000005
3477,PAYMENT0003478,ACC0022277,BRW0011422,2026-06-09 12:22:41,TXN0000000027,43010.39,SUCCESS,NACH,VND0000010
5306,PAYMENT0005307,ACC0003975,BRW0007408,2026-05-24 14:49:16,TXN0000000032,18086.24,PENDING,CARD,VND0000003
2884,PAYMENT0002885,ACC0021199,BRW0008425,2026-05-03 23:45:49,TXN0000000032,104969.97,SUCCESS,NACH,VND0000007
21182,PAYMENT0021183,ACC0011292,BRW0006242,2026-04-10 12:10:27,TXN0000000032,15714.97,SUCCESS,CASH,VND0000005
2839,PAYMENT0002840,ACC0022601,BRW0006594,2026-02-01 22:47:46,TXN0000000113,1139.66,SUCCESS,CARD,VND0000003
13292,PAYMENT0013293,ACC0019961,BRW0006917,2026-01-17 20:51:05,TXN0000000113,55554.44,REVERSED,CARD,VND0000007
18917,PAYMENT0018918,ACC0012876,BRW0010206,2026-08-03 02:17:02,TXN0000000113,101681.86,SUCCESS,CARD,VND0000010


In [17]:
#EXACT DUPLICATE PAYMENT RECORDS
payment_duplicate_mask = payments.duplicated(keep=False)

exact_duplicate_payments = (
    payments[payment_duplicate_mask]
    .sort_values(list(payments.columns))
    .copy()
)

print("Total payment rows:", len(payments))
print("Rows belonging to exact duplicate records:", len(exact_duplicate_payments))
print(
    "Unique exact-duplicate groups:",
    exact_duplicate_payments.drop_duplicates().shape[0]
)

display(exact_duplicate_payments.head(30))

Total payment rows: 25500
Rows belonging to exact duplicate records: 972
Unique exact-duplicate groups: 486


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
25035,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
25408,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
25481,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
25099,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
25330,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012


In [18]:
#DUPLICATE PAYMENT IDs
duplicate_payment_id_counts = (
    payments["payment_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Unique duplicated payment IDs:", len(duplicate_payment_id_counts))
print("Rows belonging to duplicated payment IDs:", duplicate_payment_id_counts.sum())

display(
    duplicate_payment_id_counts
    .rename_axis("payment_id")
    .reset_index(name="record_count")
    .head(30)
)

Unique duplicated payment IDs: 500
Rows belonging to duplicated payment IDs: 1000


,payment_id,record_count
0,PAYMENT0000056,2
1,PAYMENT0000076,2
2,PAYMENT0000108,2
3,PAYMENT0000149,2
4,PAYMENT0000199,2
5,PAYMENT0000255,2
6,PAYMENT0000263,2
7,PAYMENT0000311,2
8,PAYMENT0000552,2
9,PAYMENT0000553,2


In [19]:
#PAYMENT ID CONFLICT CHECK
duplicated_id_rows = payments[
    payments["payment_id"].isin(duplicate_payment_id_counts.index)
].copy()

payment_id_conflicts = (
    duplicated_id_rows
    .groupby("payment_id")
    .agg(
        record_count=("payment_id", "size"),
        unique_accounts=("account_id", "nunique"),
        unique_borrowers=("borrower_id", "nunique"),
        unique_timestamps=("event_at", "nunique"),
        unique_references=("payment_reference", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_statuses=("payment_status", "nunique"),
        unique_methods=("payment_method", "nunique"),
        unique_providers=("provider_id", "nunique")
    )
    .reset_index()
)

display(payment_id_conflicts.head(30))


,payment_id,record_count,unique_accounts,unique_borrowers,unique_timestamps,unique_references,unique_amounts,unique_statuses,unique_methods,unique_providers
0,PAYMENT0000056,2,1,1,1,1,1,1,1,1
1,PAYMENT0000076,2,1,1,1,1,1,1,1,1
2,PAYMENT0000108,2,1,1,1,1,1,1,1,1
3,PAYMENT0000149,2,1,1,1,1,1,1,1,1
4,PAYMENT0000199,2,1,1,1,1,1,1,1,1
5,PAYMENT0000255,2,1,1,1,1,1,1,1,1
6,PAYMENT0000263,2,1,1,1,1,1,1,1,1
7,PAYMENT0000311,2,1,1,1,1,1,1,1,1
8,PAYMENT0000552,2,1,1,1,1,1,1,1,1
9,PAYMENT0000553,2,1,1,1,1,1,1,1,1


In [21]:
#PAYMENT REFERENCE CONFLICT CHECK



reference_conflicts = (
    payments[payments["payment_reference"].notna()]
    .groupby("payment_reference")
    .agg(
        record_count=("payment_reference", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        unique_borrowers=("borrower_id", "nunique"),
        unique_timestamps=("event_at", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_statuses=("payment_status", "nunique"),
        unique_methods=("payment_method", "nunique"),
        unique_providers=("provider_id", "nunique")
    )
    .reset_index()
)

repeated_reference_analysis = reference_conflicts[
    reference_conflicts["record_count"] > 1
].copy()

print(
    "Repeated payment references:",
    len(repeated_reference_analysis)
)

print(
    "Repeated references with conflicting amount:",
    (
        repeated_reference_analysis["unique_amounts"] > 1
    ).sum()
)

print(
    "Repeated references with conflicting status:",
    (
        repeated_reference_analysis["unique_statuses"] > 1
    ).sum()
)

print(
    "Repeated references linked to multiple accounts:",
    (
        repeated_reference_analysis["unique_accounts"] > 1
    ).sum()
)

display(repeated_reference_analysis.head(30))


Repeated payment references: 3745
Repeated references with conflicting amount: 3407
Repeated references with conflicting status: 1681
Repeated references linked to multiple accounts: 3407


,payment_reference,record_count,unique_payment_ids,unique_accounts,unique_borrowers,unique_timestamps,unique_amounts,unique_statuses,unique_methods,unique_providers
0,TXN0000000009,2,1,1,1,1,1,1,1,1
5,TXN0000000027,2,2,2,2,2,2,2,2,2
8,TXN0000000032,3,3,3,3,3,3,2,3,3
28,TXN0000000113,3,3,3,3,3,3,2,1,3
35,TXN0000000132,2,2,2,2,2,2,1,2,2
36,TXN0000000134,2,2,2,2,2,2,1,2,2
39,TXN0000000154,2,2,2,2,2,2,1,1,2
50,TXN0000000186,2,2,2,2,2,2,1,1,2
53,TXN0000000196,2,2,2,2,2,2,1,2,2
58,TXN0000000212,2,2,2,2,2,2,1,2,2


In [22]:
#RAW SUCCESSFUL RECOVERY
raw_success = payments.loc[
    payments["payment_status"].eq("SUCCESS"),
    "amount"
].sum()

raw_success_count = (
    payments["payment_status"].eq("SUCCESS").sum()
)

print(f"Raw successful payment records: {raw_success_count:,}")
print(f"Raw successful recovery: ₹{raw_success:,.2f}")

Raw successful payment records: 17,880
Raw successful recovery: ₹1,341,485,926.33


In [23]:
#EXACT DUPLICATE SUCCESSFUL PAYMENTS
exact_duplicate_success = payments[
    payments.duplicated(keep=False)
    & payments["payment_status"].eq("SUCCESS")
].copy()

print(
    "Rows belonging to exact duplicate successful records:",
    len(exact_duplicate_success)
)

print(
    "Amount represented by all rows in duplicate groups:",
    f"₹{exact_duplicate_success['amount'].sum():,.2f}"
)

display(exact_duplicate_success.head(30))

Rows belonging to exact duplicate successful records: 670
Amount represented by all rows in duplicate groups: ₹50,022,924.38


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
600,PAYMENT0000601,ACC0028466,BRW0010885,2026-03-16 01:55:27,TXN0000048925,77126.60,SUCCESS,NETBANKING,VND0000006
622,PAYMENT0000623,ACC0011624,BRW0006699,2026-03-06 09:11:35,TXN0000060332,104701.16,SUCCESS,NETBANKING,VND0000003
723,PAYMENT0000724,ACC0017344,BRW0005581,2026-02-27 02:56:07,TXN0000046770,132663.11,SUCCESS,CARD,VND0000007
793,PAYMENT0000794,ACC0002009,BRW0002056,2026-05-07 03:35:34,TXN0000052839,22995.56,SUCCESS,NACH,VND0000008
848,PAYMENT0000849,ACC0024760,BRW0006367,2026-03-12 03:09:10,TXN0000012213,100942.27,SUCCESS,CASH,VND0000011
849,PAYMENT0000850,ACC0011675,BRW0010851,2026-01-30 19:24:32,TXN0000035005,50746.08,SUCCESS,CASH,VND0000013


In [24]:
#EXCESS RECOVERY FROM EXACT DUPLICATES



exact_duplicate_groups = (
    payments
    .groupby(list(payments.columns), dropna=False)
    .agg(
        duplicate_count=("payment_id", "size")
    )
    .reset_index()
)

exact_duplicate_groups["excess_records"] = (
    exact_duplicate_groups["duplicate_count"] - 1
)

exact_duplicate_groups["excess_amount"] = (
    exact_duplicate_groups["amount"]
    * exact_duplicate_groups["excess_records"]
)

exact_duplicate_success_groups = exact_duplicate_groups[
    exact_duplicate_groups["payment_status"].eq("SUCCESS")
]

excess_exact_duplicate_recovery = (
    exact_duplicate_success_groups["excess_amount"].sum()
)

print(
    f"Excess recovery from exact duplicate SUCCESS records: "
    f"₹{excess_exact_duplicate_recovery:,.2f}"
)

Excess recovery from exact duplicate SUCCESS records: ₹25,011,462.19


In [25]:
#DEDUPLICATED PAYMENT BENCHMARK



payments_exact_dedup = payments.drop_duplicates(
    keep="first"
).copy()

dedup_success = payments_exact_dedup.loc[
    payments_exact_dedup["payment_status"].eq("SUCCESS"),
    "amount"
].sum()

dedup_success_count = (
    payments_exact_dedup["payment_status"].eq("SUCCESS").sum()
)

print(f"Raw payment rows: {len(payments):,}")
print(f"After exact-row deduplication: {len(payments_exact_dedup):,}")

print(f"\nRaw SUCCESS records: {raw_success_count:,}")
print(f"Deduplicated SUCCESS records: {dedup_success_count:,}")

print(f"\nRaw SUCCESS recovery: ₹{raw_success:,.2f}")
print(f"Exact-deduplicated SUCCESS recovery: ₹{dedup_success:,.2f}")

print(
    f"Excess recovery removed: "
    f"₹{raw_success - dedup_success:,.2f}"
)

print(
    f"Recovery overstatement: "
    f"{(raw_success - dedup_success) / raw_success * 100:.2f}%"
)


Raw payment rows: 25,500
After exact-row deduplication: 25,014

Raw SUCCESS records: 17,880
Deduplicated SUCCESS records: 17,545

Raw SUCCESS recovery: ₹1,341,485,926.33
Exact-deduplicated SUCCESS recovery: ₹1,316,474,464.14
Excess recovery removed: ₹25,011,462.19
Recovery overstatement: 1.86%


In [26]:
#MONTHLY RAW VS EXACT-DEDUPLICATED RECOVERY



payments["event_at_parsed"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

payments_exact_dedup["event_at_parsed"] = pd.to_datetime(
    payments_exact_dedup["event_at"],
    errors="coerce"
)

raw_monthly = (
    payments[
        payments["payment_status"].eq("SUCCESS")
    ]
    .assign(month=lambda x: x["event_at_parsed"].dt.to_period("M"))
    .groupby("month", as_index=False)
    .agg(
        raw_recovery=("amount", "sum"),
        raw_success_records=("payment_id", "count")
    )
)

dedup_monthly = (
    payments_exact_dedup[
        payments_exact_dedup["payment_status"].eq("SUCCESS")
    ]
    .assign(month=lambda x: x["event_at_parsed"].dt.to_period("M"))
    .groupby("month", as_index=False)
    .agg(
        dedup_recovery=("amount", "sum"),
        dedup_success_records=("payment_id", "count")
    )
)

monthly_payment_integrity = raw_monthly.merge(
    dedup_monthly,
    on="month",
    how="outer"
)

monthly_payment_integrity["recovery_difference"] = (
    monthly_payment_integrity["raw_recovery"]
    - monthly_payment_integrity["dedup_recovery"]
)

monthly_payment_integrity["overstatement_pct"] = (
    monthly_payment_integrity["recovery_difference"]
    / monthly_payment_integrity["raw_recovery"]
    * 100
)

monthly_payment_integrity


,month,raw_recovery,raw_success_records,dedup_recovery,dedup_success_records,recovery_difference,overstatement_pct
0,2026-01,1.911333e+08,2515,1.872291e+08,2464,3904156.83,2.042636
1,2026-02,1.740973e+08,2315,1.702796e+08,2270,3817672.99,2.192839
2,2026-03,1.932334e+08,2576,1.891903e+08,2527,4043057.71,2.092318
3,2026-04,1.784270e+08,2455,1.752289e+08,2407,3198086.88,1.792378
4,2026-05,1.870481e+08,2486,1.843355e+08,2450,2712660.68,1.450247
5,2026-06,1.787245e+08,2410,1.758534e+08,2369,2871043.60,1.606407
6,2026-07,1.902788e+08,2486,1.872478e+08,2442,3031010.88,1.592931
7,2026-08,4.854347e+07,637,4.710970e+07,616,1433772.62,2.953585


In [27]:
# ATTRIBUTION-RELATED SCHEMA INSPECTION
attribution_tables = [
    "payments",
    "calls",
    "call_attempts",
    "daily_targeting",
    "campaigns"
]

for table_name in attribution_tables:
    print("\n" + "=" * 70)
    print(f"{table_name}")
    print("=" * 70)

    df = datasets[table_name]

    print("Columns:")
    print(df.columns.tolist())

    print("\nSample:")
    display(df.head(3))


payments
Columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

Sample:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013



calls
Columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

Sample:


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
0,CALL0000001,ACC0011505,BRW0007139,2026-07-15 15:36:22,AGT0000955,CMP0000015,OUTBOUND,VND0000013,NO_ANSWER,601,Asia/Dubai
1,CALL0000002,ACC0002025,BRW0006253,2026-06-10 06:48:27,AGT0000853,CMP0000060,OUTBOUND,VND0000001,ANSWERED,224,Asia/Kolkata
2,CALL0000003,ACC0013375,BRW0007663,2026-04-07 00:35:35,AGT0000525,CMP0000060,OUTBOUND,VND0000008,FAILED,7,Asia/Dubai



call_attempts
Columns:
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'attempt_no', 'vendor_id', 'attempt_status']

Sample:


,attempt_id,account_id,borrower_id,event_at,call_id,agent_id,attempt_no,vendor_id,attempt_status
0,ATTEMPT0000001,ACC0009527,BRW0000705,2026-03-10 21:31:13,CALL0019052,AGT0000685,5,VND0000013,FAILED
1,ATTEMPT0000002,ACC0007960,BRW0002732,2026-05-01 20:55:02,CALL0011950,AGT0000472,4,VND0000006,CONNECTED
2,ATTEMPT0000003,ACC0016476,BRW0002174,2026-06-05 20:57:37,CALL0017411,AGT0000035,1,VND0000015,RINGING



daily_targeting
Columns:
['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']

Sample:


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED



campaigns
Columns:
['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

Sample:


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51


In [28]:
#BUILD PAYMENT-INTERACTION TIMELINE



payments = datasets["payments"].copy()
calls = datasets["calls"].copy()
daily_targeting = datasets["daily_targeting"].copy()
campaigns = datasets["campaigns"].copy()

# Parse timestamps
payments["event_at_dt"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

calls["event_at_dt"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

daily_targeting["target_date_dt"] = pd.to_datetime(
    daily_targeting["target_date"],
    errors="coerce"
)

campaigns["start_at_dt"] = pd.to_datetime(
    campaigns["start_at"],
    errors="coerce"
)

campaigns["end_at_dt"] = pd.to_datetime(
    campaigns["end_at"],
    errors="coerce"
)

print("Payments:", len(payments))
print("Calls:", len(calls))
print("Daily targeting records:", len(daily_targeting))
print("Campaigns:", len(campaigns))


Payments: 25500
Calls: 91350
Daily targeting records: 45000
Campaigns: 120


In [29]:
#LATEST CALL BEFORE PAYMENT


payment_events = payments[
    payments["payment_status"].eq("SUCCESS")
].copy()

payment_events = payment_events.sort_values(
    ["account_id", "event_at_dt"]
)

call_events = calls[
    calls["event_at_dt"].notna()
].sort_values(
    ["account_id", "event_at_dt"]
)

latest_call_before_payment = pd.merge_asof(
    payment_events.sort_values("event_at_dt"),
    call_events.sort_values("event_at_dt"),
    by="account_id",
    left_on="event_at_dt",
    right_on="event_at_dt",
    direction="backward",
    suffixes=("_payment", "_call")
)

print(
    "Successful payments:",
    len(payment_events)
)

print(
    "Successful payments with a previous call:",
    latest_call_before_payment["call_id"].notna().sum()
)

print(
    "Successful payments without a previous call:",
    latest_call_before_payment["call_id"].isna().sum()
)

display(
    latest_call_before_payment[
        [
            "payment_id",
            "account_id",
            "event_at_dt",
            "amount",
            "call_id",
            "campaign_id",
            "agent_id",
            "call_status",
            "timezone"
        ]
    ].head(20)
)

Successful payments: 17880
Successful payments with a previous call: 12173
Successful payments without a previous call: 5707


,payment_id,account_id,event_at_dt,amount,call_id,campaign_id,agent_id,call_status,timezone
0,PAYMENT0008675,ACC0014142,2026-01-01 00:14:40,99867.36,NaN,NaN,NaN,NaN,NaN
1,PAYMENT0002693,ACC0014449,2026-01-01 00:41:58,101870.25,NaN,NaN,NaN,NaN,NaN
2,PAYMENT0024821,ACC0018361,2026-01-01 00:50:36,87906.80,NaN,NaN,NaN,NaN,NaN
3,PAYMENT0011191,ACC0004322,2026-01-01 01:02:22,94150.20,NaN,NaN,NaN,NaN,NaN
4,PAYMENT0008443,ACC0005569,2026-01-01 01:21:07,81954.29,NaN,NaN,NaN,NaN,NaN
5,PAYMENT0020533,ACC0002164,2026-01-01 01:21:45,74524.80,NaN,NaN,NaN,NaN,NaN
6,PAYMENT0010455,ACC0002916,2026-01-01 01:42:55,14211.83,NaN,NaN,NaN,NaN,NaN
7,PAYMENT0017955,ACC0016208,2026-01-01 02:12:07,130720.85,NaN,NaN,NaN,NaN,NaN
8,PAYMENT0002337,ACC0028378,2026-01-01 03:56:34,33260.86,NaN,NaN,NaN,NaN,NaN
9,PAYMENT0008918,ACC0025342,2026-01-01 04:25:40,129926.36,NaN,NaN,NaN,NaN,NaN


In [31]:

#  TIME FROM LAST CALL TO PAYMENT


# Convert the timestamp columns produced by merge_asof
latest_call_before_payment["payment_timestamp"] = pd.to_datetime(
    latest_call_before_payment["event_at_payment"],
    errors="coerce"
)

latest_call_before_payment["call_timestamp"] = pd.to_datetime(
    latest_call_before_payment["event_at_call"],
    errors="coerce"
)

# Calculate elapsed time between the latest previous call and payment
latest_call_before_payment["hours_since_last_call"] = (
    latest_call_before_payment["payment_timestamp"]
    - latest_call_before_payment["call_timestamp"]
).dt.total_seconds() / 3600

print("Time difference calculated successfully.")

display(
    latest_call_before_payment[
        [
            "payment_id",
            "account_id",
            "payment_timestamp",
            "call_timestamp",
            "campaign_id",
            "hours_since_last_call"
        ]
    ].head(20)
)

Time difference calculated successfully.


,payment_id,account_id,payment_timestamp,call_timestamp,campaign_id,hours_since_last_call
0,PAYMENT0008675,ACC0014142,2026-01-01 00:14:40,NaT,NaN,NaN
1,PAYMENT0002693,ACC0014449,2026-01-01 00:41:58,NaT,NaN,NaN
2,PAYMENT0024821,ACC0018361,2026-01-01 00:50:36,NaT,NaN,NaN
3,PAYMENT0011191,ACC0004322,2026-01-01 01:02:22,NaT,NaN,NaN
4,PAYMENT0008443,ACC0005569,2026-01-01 01:21:07,NaT,NaN,NaN
5,PAYMENT0020533,ACC0002164,2026-01-01 01:21:45,NaT,NaN,NaN
6,PAYMENT0010455,ACC0002916,2026-01-01 01:42:55,NaT,NaN,NaN
7,PAYMENT0017955,ACC0016208,2026-01-01 02:12:07,NaT,NaN,NaN
8,PAYMENT0002337,ACC0028378,2026-01-01 03:56:34,NaT,NaN,NaN
9,PAYMENT0008918,ACC0025342,2026-01-01 04:25:40,NaT,NaN,NaN


In [32]:
#ATTRIBUTION-WINDOW ANALYSIS




# Work only with successful payments that have a valid timestamp
attribution_base = latest_call_before_payment[
    latest_call_before_payment["payment_timestamp"].notna()
    & latest_call_before_payment["call_timestamp"].notna()
].copy()

# Calculate hours between the call and payment
attribution_base["hours_since_last_call"] = (
    attribution_base["payment_timestamp"]
    - attribution_base["call_timestamp"]
).dt.total_seconds() / 3600

# Windows required for comparison
windows = [0, 24, 72, 168]

attribution_results = []

total_successful_payments = len(payment_events)

for hours in windows:
    if hours == 0:
        eligible = attribution_base[
            attribution_base["hours_since_last_call"] >= 0
        ]
        window_name = "same_day"
    else:
        eligible = attribution_base[
            (attribution_base["hours_since_last_call"] >= 0)
            & (attribution_base["hours_since_last_call"] <= hours)
        ]
        window_name = f"{hours}_hours"

    attributed_recovery = eligible["amount"].sum()
    attributed_payments = eligible["payment_id"].nunique()

    attribution_results.append({
        "attribution_window": window_name,
        "window_hours": hours,
        "attributed_payment_count": attributed_payments,
        "attributed_recovery": attributed_recovery,
        "pct_successful_payments_attributed": (
            attributed_payments / total_successful_payments * 100
        ),
        "pct_successful_recovery_attributed": (
            attributed_recovery / payment_events["amount"].sum() * 100
        )
    })

attribution_window_df = pd.DataFrame(attribution_results)

attribution_window_df

,attribution_window,window_hours,attributed_payment_count,attributed_recovery,pct_successful_payments_attributed,pct_successful_recovery_attributed
0,same_day,0,11926,9.137639e+08,66.700224,68.115805
1,24_hours,24,250,1.808055e+07,1.398210,1.347800
2,72_hours,72,724,5.483415e+07,4.049217,4.087568
3,168_hours,168,1583,1.216553e+08,8.853468,9.068697


In [33]:
#PAYMENT-TO-LAST-CALL TIME DISTRIBUTION

time_summary = attribution_base[
    attribution_base["hours_since_last_call"] >= 0
]["hours_since_last_call"].describe()

time_summary

count    12173.000000
mean      1053.988435
std        923.307517
min          0.303056
25%        343.200000
50%        793.710000
75%       1500.234167
max       5117.720278
Name: hours_since_last_call, dtype: float64

In [34]:
# CAMPAIGN ATTRIBUTION UNDER 7-DAY WINDOW
seven_day_attribution = attribution_base[
    (attribution_base["hours_since_last_call"] >= 0)
    & (attribution_base["hours_since_last_call"] <= 168)
].copy()

campaign_attribution = (
    seven_day_attribution
    .groupby("campaign_id", dropna=False)
    .agg(
        successful_payments=("payment_id", "nunique"),
        recovery=("amount", "sum"),
        accounts=("account_id", "nunique")
    )
    .reset_index()
    .sort_values("recovery", ascending=False)
)

campaign_attribution

,campaign_id,successful_payments,recovery,accounts
70,CMP0000071,21,2014395.25,20
116,CMP0000117,19,1876268.32,18
46,CMP0000047,20,1631284.56,19
94,CMP0000095,15,1611801.73,14
32,CMP0000033,19,1567885.83,19
...,...,...,...,...
43,CMP0000044,7,562400.85,7
45,CMP0000046,7,501754.33,7
26,CMP0000027,8,462051.90,8
29,CMP0000030,7,408240.59,7


In [35]:
#TIMEZONE FORENSICS — TIMEZONE DISTRIBUTION
calls = datasets["calls"].copy()

print("Timezone values:")
display(
    calls["timezone"]
    .value_counts(dropna=False)
    .rename_axis("timezone")
    .reset_index(name="call_count")
)

Timezone values:


,timezone,call_count
0,Asia/Kolkata,30485
1,Asia/Dubai,30464
2,UTC,30401


In [36]:
#CALL TIMESTAMP PARSING
calls["event_at_parsed"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

print("Total calls:", len(calls))
print(
    "Valid timestamps:",
    calls["event_at_parsed"].notna().sum()
)
print(
    "Invalid timestamps:",
    calls["event_at_parsed"].isna().sum()
)

display(
    calls[
        ["event_at", "timezone", "event_at_parsed"]
    ].head(10)
)

Total calls: 91350
Valid timestamps: 91350
Invalid timestamps: 0


,event_at,timezone,event_at_parsed
0,2026-07-15 15:36:22,Asia/Dubai,2026-07-15 15:36:22
1,2026-06-10 06:48:27,Asia/Kolkata,2026-06-10 06:48:27
2,2026-04-07 00:35:35,Asia/Dubai,2026-04-07 00:35:35
3,2026-02-12 14:16:57,Asia/Dubai,2026-02-12 14:16:57
4,2026-05-24 15:33:12,Asia/Kolkata,2026-05-24 15:33:12
5,2026-02-04 11:37:08,Asia/Dubai,2026-02-04 11:37:08
6,2026-02-10 10:59:05,Asia/Kolkata,2026-02-10 10:59:05
7,2026-06-14 23:37:56,Asia/Dubai,2026-06-14 23:37:56
8,2026-03-17 16:24:42,UTC,2026-03-17 16:24:42
9,2026-06-24 04:26:34,UTC,2026-06-24 04:26:34


In [37]:
#TIMESTAMP FORMAT INSPECTION
display(
    calls[
        ["event_at", "timezone"]
    ].head(30)
)

print("\nUnique timestamp formats / examples:")
for value in calls["event_at"].dropna().astype(str).head(20):
    print(value)

,event_at,timezone
0,2026-07-15 15:36:22,Asia/Dubai
1,2026-06-10 06:48:27,Asia/Kolkata
2,2026-04-07 00:35:35,Asia/Dubai
3,2026-02-12 14:16:57,Asia/Dubai
4,2026-05-24 15:33:12,Asia/Kolkata
5,2026-02-04 11:37:08,Asia/Dubai
6,2026-02-10 10:59:05,Asia/Kolkata
7,2026-06-14 23:37:56,Asia/Dubai
8,2026-03-17 16:24:42,UTC
9,2026-06-24 04:26:34,UTC



Unique timestamp formats / examples:
2026-07-15 15:36:22
2026-06-10 06:48:27
2026-04-07 00:35:35
2026-02-12 14:16:57
2026-05-24 15:33:12
2026-02-04 11:37:08
2026-02-10 10:59:05
2026-06-14 23:37:56
2026-03-17 16:24:42
2026-06-24 04:26:34
2026-07-23 14:50:00
2026-02-12 20:40:45
2026-05-30 17:57:19
2026-01-10 17:09:15
2026-06-13 22:35:10
2026-01-01 10:44:31
2026-01-21 16:48:34
2026-04-28 11:02:01
2026-01-05 08:28:07
2026-04-26 16:31:37


In [38]:

#LOCAL TIME VS UTC TIME



from zoneinfo import ZoneInfo

def convert_to_utc(row):
    try:
        timestamp = pd.Timestamp(row["event_at"])
        timezone = row["timezone"]

        if pd.isna(timestamp) or pd.isna(timezone):
            return pd.NaT

        # Treat supplied event_at as local time when timezone is supplied
        local_timestamp = timestamp.tz_localize(
            ZoneInfo(str(timezone)),
            ambiguous="NaT",
            nonexistent="NaT"
        )

        return local_timestamp.tz_convert("UTC")

    except Exception:
        return pd.NaT


calls["event_at_utc"] = calls.apply(
    convert_to_utc,
    axis=1
)

calls["local_hour"] = calls["event_at_parsed"].dt.hour
calls["utc_hour"] = calls["event_at_utc"].dt.hour

calls["local_date"] = calls["event_at_parsed"].dt.date
calls["utc_date"] = calls["event_at_utc"].dt.date

print(
    "Successfully converted to UTC:",
    calls["event_at_utc"].notna().sum()
)

display(
    calls[
        [
            "event_at",
            "timezone",
            "local_hour",
            "utc_hour",
            "local_date",
            "utc_date"
        ]
    ].head(20)
)


Successfully converted to UTC: 91350


,event_at,timezone,local_hour,utc_hour,local_date,utc_date
0,2026-07-15 15:36:22,Asia/Dubai,15,11,2026-07-15,2026-07-15
1,2026-06-10 06:48:27,Asia/Kolkata,6,1,2026-06-10,2026-06-10
2,2026-04-07 00:35:35,Asia/Dubai,0,20,2026-04-07,2026-04-06
3,2026-02-12 14:16:57,Asia/Dubai,14,10,2026-02-12,2026-02-12
4,2026-05-24 15:33:12,Asia/Kolkata,15,10,2026-05-24,2026-05-24
5,2026-02-04 11:37:08,Asia/Dubai,11,7,2026-02-04,2026-02-04
6,2026-02-10 10:59:05,Asia/Kolkata,10,5,2026-02-10,2026-02-10
7,2026-06-14 23:37:56,Asia/Dubai,23,19,2026-06-14,2026-06-14
8,2026-03-17 16:24:42,UTC,16,16,2026-03-17,2026-03-17
9,2026-06-24 04:26:34,UTC,4,4,2026-06-24,2026-06-24


In [39]:
#TIMEZONE IMPACT



timezone_analysis = calls[
    calls["event_at_utc"].notna()
].copy()

timezone_analysis["hour_changed"] = (
    timezone_analysis["local_hour"]
    != timezone_analysis["utc_hour"]
)

timezone_analysis["date_changed"] = (
    timezone_analysis["local_date"]
    != timezone_analysis["utc_date"]
)

timezone_summary = pd.DataFrame({
    "metric": [
        "Calls with valid timezone conversion",
        "Calls whose hour changes",
        "Calls whose calendar date changes"
    ],
    "count": [
        len(timezone_analysis),
        timezone_analysis["hour_changed"].sum(),
        timezone_analysis["date_changed"].sum()
    ]
})

timezone_summary["percentage"] = (
    timezone_summary["count"]
    / len(timezone_analysis)
    * 100
)

timezone_summary


,metric,count,percentage
0,Calls with valid timezone conversion,91350,100.000000
1,Calls whose hour changes,60949,66.720307
2,Calls whose calendar date changes,12097,13.242474


In [40]:
# TIMEZONE IMPACT BY SUPPLIED TIMEZONE
timezone_breakdown = (
    timezone_analysis
    .groupby("timezone")
    .agg(
        calls=("call_id", "count"),
        hour_changed=("hour_changed", "sum"),
        date_changed=("date_changed", "sum")
    )
    .reset_index()
)

timezone_breakdown["hour_changed_pct"] = (
    timezone_breakdown["hour_changed"]
    / timezone_breakdown["calls"]
    * 100
)

timezone_breakdown["date_changed_pct"] = (
    timezone_breakdown["date_changed"]
    / timezone_breakdown["calls"]
    * 100
)

timezone_breakdown

,timezone,calls,hour_changed,date_changed,hour_changed_pct,date_changed_pct
0,Asia/Dubai,30464,30464,5039,100.0,16.540835
1,Asia/Kolkata,30485,30485,7058,100.0,23.152370
2,UTC,30401,0,0,0.0,0.000000


In [41]:
#VENDOR / TELEPHONY SCHEMA INSPECTION
vendor_telephony = datasets["vendor_telephony"].copy()
call_dispositions = datasets["call_dispositions"].copy()
calls = datasets["calls"].copy()

print("VENDOR_TELEPHONY")
print("=" * 70)
print(vendor_telephony.columns.tolist())
display(vendor_telephony.head())

print("\nCALL_DISPOSITIONS")
print("=" * 70)
print(call_dispositions.columns.tolist())
display(call_dispositions.head())

VENDOR_TELEPHONY
['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']


,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3
3,VND0000004,Twilio,VAC321507,UTC,ACTIVE,v1
4,VND0000005,Twilio,VAC976733,Asia/Kolkata,ACTIVE,v1



CALL_DISPOSITIONS
['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version']


,disposition_id,account_id,borrower_id,event_at,call_id,agent_id,disposition_code,disposition_version
0,DISPOSITION0000001,ACC0029208,BRW0006282,2026-03-07 17:30:48,CALL0032272,AGT0000759,CALLBACK,v2
1,DISPOSITION0000002,ACC0009584,BRW0000540,2026-04-04 11:59:50,CALL0053880,AGT0000085,CALLBACK,legacy
2,DISPOSITION0000003,ACC0003386,BRW0010257,2026-03-11 01:25:13,CALL0048622,AGT0000319,PROMISE_TO_PAY,v1
3,DISPOSITION0000004,ACC0027899,BRW0006534,2026-03-17 14:49:43,CALL0072651,AGT0000189,NO_CONTACT,legacy
4,DISPOSITION0000005,ACC0024345,BRW0001918,2026-03-07 06:07:26,CALL0036866,AGT0000647,PTP,legacy


In [42]:
#TELEPHONY VENDOR DISTRIBUTION
print("Vendor distribution in calls:")

display(
    calls["vendor_id"]
    .value_counts(dropna=False)
    .rename_axis("vendor_id")
    .reset_index(name="call_count")
)

Vendor distribution in calls:


,vendor_id,call_count
0,VND0000010,6248
1,VND0000011,6177
2,VND0000005,6153
3,VND0000015,6150
4,VND0000007,6138
5,VND0000014,6132
6,VND0000001,6128
7,VND0000012,6094
8,VND0000003,6068
9,VND0000009,6061


In [43]:
# DISPOSITION CODE DISTRIBUTION
print("Call disposition columns:")
print(call_dispositions.columns.tolist())

display(
    call_dispositions.head(20)
)

Call disposition columns:
['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version']


,disposition_id,account_id,borrower_id,event_at,call_id,agent_id,disposition_code,disposition_version
0,DISPOSITION0000001,ACC0029208,BRW0006282,2026-03-07 17:30:48,CALL0032272,AGT0000759,CALLBACK,v2
1,DISPOSITION0000002,ACC0009584,BRW0000540,2026-04-04 11:59:50,CALL0053880,AGT0000085,CALLBACK,legacy
2,DISPOSITION0000003,ACC0003386,BRW0010257,2026-03-11 01:25:13,CALL0048622,AGT0000319,PROMISE_TO_PAY,v1
3,DISPOSITION0000004,ACC0027899,BRW0006534,2026-03-17 14:49:43,CALL0072651,AGT0000189,NO_CONTACT,legacy
4,DISPOSITION0000005,ACC0024345,BRW0001918,2026-03-07 06:07:26,CALL0036866,AGT0000647,PTP,legacy
5,DISPOSITION0000006,ACC0000656,BRW0010734,2026-05-04 06:12:23,CALL0078106,AGT0000534,CALLBACK,v2
6,DISPOSITION0000007,ACC0015494,BRW0002011,2026-01-09 16:25:53,CALL0078570,AGT0000755,PAID,v1
7,DISPOSITION0000008,ACC0009000,BRW0005628,2026-04-03 06:08:13,CALL0042609,AGT0000856,PROMISE_TO_PAY,v1
8,DISPOSITION0000009,ACC0021766,BRW0011962,2026-07-03 22:35:05,CALL0009101,AGT0000953,PROMISE_TO_PAY,v1
9,DISPOSITION0000010,ACC0004669,BRW0008936,2026-06-23 04:59:52,CALL0084384,AGT0000413,NO_CONTACT,v2


In [44]:
#AGENT IDENTITY FORENSICS
agents = datasets["agents"].copy()
agent_sessions = datasets["agent_sessions"].copy()

print("AGENTS COLUMNS")
print("=" * 70)
print(agents.columns.tolist())

display(agents.head())

print("\nAGENT_SESSIONS COLUMNS")
print("=" * 70)
print(agent_sessions.columns.tolist())

display(agent_sessions.head())


AGENTS COLUMNS
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
0,AGT0000760,EMP00323,Vikram Shah,VND0000012,T3,INACTIVE,2025-06-19 20:30:53,2025-12-11 03:40:22
1,AGT0000171,EMP00079,Pooja Nair,VND0000012,T1,INACTIVE,2025-08-26 11:06:26,2025-10-11 08:30:33
2,AGT0000766,EMP00030,Sneha Das,VND0000002,FIELD,SUSPENDED,2024-05-30 11:23:06,2026-05-16 03:02:02
3,AGT0000031,EMP00334,Aarav Sharma,VND0000007,T1,INACTIVE,2024-10-11 18:58:28,2025-10-16 13:31:21
4,AGT0000669,EMP00014,Aarav Sharma,VND0000001,T3,SUSPENDED,2024-09-01 19:55:06,2026-07-04 03:33:11



AGENT_SESSIONS COLUMNS
['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']


,session_id,agent_id,login_at,channel,device_id,timezone,logout_at
0,SES0000001,AGT0000117,2026-08-06 13:33:27,VOICE,DEV00243,Asia/Kolkata,2026-08-06 18:41:13
1,SES0000002,AGT0000847,2026-05-27 02:02:28,FIELD,DEV01359,UTC,2026-05-27 09:02:31
2,SES0000003,AGT0000989,2026-08-01 20:33:44,FIELD,DEV01432,Asia/Kolkata,2026-08-02 05:53:43
3,SES0000004,AGT0000961,2026-01-13 19:50:52,FIELD,DEV00402,Asia/Kolkata,2026-01-13 22:20:49
4,SES0000005,AGT0000280,2026-03-03 15:05:04,VOICE,DEV01288,UTC,2026-03-03 18:19:09


In [45]:
#AGENT ID UNIQUENESS
agent_identity_summary = []

for column in agents.columns:

    if "agent" in column.lower() or "employee" in column.lower():
        agent_identity_summary.append({
            "column": column,
            "rows": len(agents),
            "unique_values": agents[column].nunique(dropna=True),
            "duplicate_rows": agents[column].duplicated().sum(),
            "missing_values": agents[column].isna().sum()
        })

agent_identity_summary_df = pd.DataFrame(
    agent_identity_summary
)

agent_identity_summary_df

,column,rows,unique_values,duplicate_rows,missing_values
0,agent_id,30000,1000,29000,0
1,employee_code,30000,1099,28901,0
2,agent_name,30000,10,29990,0


In [46]:
#AGENT ID TO EMPLOYEE MAPPING
agent_columns = [
    col for col in agents.columns
    if "agent" in col.lower()
    or "employee" in col.lower()
]

agent_mapping = (
    agents
    .groupby("agent_id", dropna=False)
    .agg(
        record_count=("agent_id", "size"),
        **{
            f"unique_{col}": (col, "nunique")
            for col in agent_columns
            if col != "agent_id"
        }
    )
    .reset_index()
)

agent_mapping

,agent_id,record_count,unique_employee_code,unique_agent_name
0,AGT0000001,23,23,10
1,AGT0000002,23,23,10
2,AGT0000003,28,28,10
3,AGT0000004,28,28,10
4,AGT0000005,29,29,10
...,...,...,...,...
995,AGT0000996,26,26,10
996,AGT0000997,42,40,10
997,AGT0000998,32,32,10
998,AGT0000999,27,27,8


In [47]:
#CONFLICTING AGENT IDENTITIES
conflict_columns = [
    col for col in agent_mapping.columns
    if col.startswith("unique_")
]

conflicting_agents = agent_mapping[
    (agent_mapping[conflict_columns] > 1).any(axis=1)
].copy()

print(
    "Agent IDs with at least one conflicting identity attribute:",
    len(conflicting_agents)
)

display(conflicting_agents.head(30))

Agent IDs with at least one conflicting identity attribute: 1000


,agent_id,record_count,unique_employee_code,unique_agent_name
0,AGT0000001,23,23,10
1,AGT0000002,23,23,10
2,AGT0000003,28,28,10
3,AGT0000004,28,28,10
4,AGT0000005,29,29,10
5,AGT0000006,37,37,10
6,AGT0000007,30,29,10
7,AGT0000008,29,29,10
8,AGT0000009,25,24,10
9,AGT0000010,26,26,10


In [48]:
#ACTUAL AGENT IDENTITY CONFLICTS
conflicting_agent_ids = conflicting_agents["agent_id"].tolist()

agent_conflict_records = agents[
    agents["agent_id"].isin(conflicting_agent_ids)
].copy()

display(
    agent_conflict_records.sort_values("agent_id").head(50)
)

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
14127,AGT0000001,EMP00308,Rohan Patel,VND0000008,FIELD,INACTIVE,2025-02-04 06:36:16,2026-03-02 00:15:29
25883,AGT0000001,EMP00404,Ananya Rao,VND0000014,T2,SUSPENDED,2025-11-13 07:54:11,2025-10-27 07:59:16
8769,AGT0000001,EMP00861,Aarav Sharma,VND0000015,T3,INACTIVE,2025-03-16 08:57:58,2026-05-11 23:40:59
19963,AGT0000001,EMP00903,Sneha Das,VND0000002,T3,INACTIVE,2024-02-21 17:11:06,2025-09-18 08:17:45
4511,AGT0000001,EMP00113,Amit Kumar,VND0000015,T1,SUSPENDED,2025-08-03 12:02:41,2026-05-03 03:37:58
7200,AGT0000001,EMP00117,Pooja Nair,VND0000009,T3,ACTIVE,2024-02-03 19:49:59,2026-04-17 04:53:24
25602,AGT0000001,EMP01073,Ananya Rao,VND0000002,FIELD,ACTIVE,2025-05-27 17:32:59,2026-06-10 08:28:37
6520,AGT0000001,EMP00632,Ananya Rao,VND0000007,DIGITAL,ACTIVE,2024-08-02 00:51:56,2025-08-30 06:00:31
278,AGT0000001,EMP01065,Neha Singh,VND0000007,T3,ACTIVE,2024-05-12 03:20:38,2026-04-14 22:45:46
28144,AGT0000001,EMP00562,Amit Kumar,VND0000015,T2,SUSPENDED,2025-01-04 00:30:22,2026-05-29 10:43:33


In [49]:
#PORTFOLIO MIX — ACCOUNT STRUCTURE
accounts = datasets["accounts"].copy()

print("Accounts shape:", accounts.shape)

print("\nAccount columns:")
print(accounts.columns.tolist())

print("\nSample records:")
display(accounts.head())

Accounts shape: (30000, 11)

Account columns:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

Sample records:


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2


In [50]:
#PORTFOLIO MIX — CATEGORICAL DIMENSIONS
categorical_columns = [
    col for col in accounts.columns
    if accounts[col].dtype == "object"
]

for col in categorical_columns:

    print("\n" + "=" * 70)
    print(col)
    print("=" * 70)

    display(
        accounts[col]
        .value_counts(dropna=False)
        .head(20)
        .rename_axis(col)
        .reset_index(name="account_count")
    )

In [51]:
# PORTFOLIO MIX — NUMERICAL DIMENSIONS
numeric_columns = accounts.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical account fields:")
print(numeric_columns)

display(
    accounts[numeric_columns].describe().T
)

Numerical account fields:
['principal_amount', 'outstanding_amount', 'dpd']


,count,mean,std,min,25%,50%,75%,max
principal_amount,30000.0,403455.521785,228521.364422,10055.11,204114.1025,403456.505,601484.3500,799968.38
outstanding_amount,30000.0,349634.511433,201660.392809,1002.67,175556.8150,349587.085,522953.6875,699963.89
dpd,30000.0,56.506433,53.970460,0.00,5.0000,45.000,90.0000,180.00


In [52]:
#ACCOUNT DATE FIELDS
date_like_columns = [
    col for col in accounts.columns
    if any(keyword in col.lower() for keyword in [
        "date", "time", "at"
    ])
]

print("Potential date/time fields:")
print(date_like_columns)

for col in date_like_columns:

    parsed = pd.to_datetime(
        accounts[col],
        errors="coerce"
    )

    print(
        f"\n{col}: "
        f"{parsed.notna().sum():,} valid | "
        f"min={parsed.min()} | "
        f"max={parsed.max()}"
    )

Potential date/time fields:
['status', 'opened_at', 'timezone']

status: 0 valid | min=NaT | max=NaT

opened_at: 30,000 valid | min=2024-01-01 00:02:27 | max=2025-11-30 23:52:36

timezone: 0 valid | min=NaT | max=NaT


C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\1378804677.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\1378804677.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(


In [53]:
#ACCOUNT TIME FIELD IDENTIFICATION
print("Account columns containing date/time information:")

account_time_candidates = [
    col for col in accounts.columns
    if any(x in col.lower() for x in [
        "date", "time", "at"
    ])
]

for col in account_time_candidates:
    print("-", col)

Account columns containing date/time information:
- status
- opened_at
- timezone


In [54]:
#ACCOUNT-MONTH POPULATION


print("Available account time candidates:")
print(account_time_candidates)

# Select the first candidate that contains a meaningful number
# of parseable dates.
account_time_column = None

for col in account_time_candidates:
    parsed = pd.to_datetime(accounts[col], errors="coerce")

    if parsed.notna().sum() > 0:
        account_time_column = col
        accounts["analysis_timestamp"] = parsed
        break

print("\nSelected time field:", account_time_column)

if account_time_column is None:
    print("No usable account date/time field was found.")
else:
    accounts["analysis_month"] = (
        accounts["analysis_timestamp"]
        .dt.to_period("M")
    )

    account_month_population = (
        accounts
        .groupby("analysis_month")
        .agg(
            account_count=("account_id", "nunique")
        )
        .reset_index()
    )

    display(account_month_population)


Available account time candidates:
['status', 'opened_at', 'timezone']

Selected time field: opened_at


C:\Users\This PC\AppData\Local\Temp\ipykernel_1672\1188622871.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(accounts[col], errors="coerce")


,analysis_month,account_count
0,2024-01,1311
1,2024-02,1299
2,2024-03,1404
3,2024-04,1263
4,2024-05,1287
5,2024-06,1219
6,2024-07,1324
7,2024-08,1358
8,2024-09,1326
9,2024-10,1362


In [57]:

#  MONTHLY PORTFOLIO MIX


account_mix_results = []

# Re-identify categorical columns from the supplied accounts data
portfolio_dimensions = [
    col for col in accounts.columns
    if accounts[col].dtype == "object"
    and col not in ["account_id"]
]

print("Portfolio dimensions found:")
print(portfolio_dimensions)

for col in portfolio_dimensions:

    # Skip identifiers / extremely high-cardinality fields
    unique_count = accounts[col].nunique(dropna=True)

    if unique_count > 50:
        continue

    temp = (
        accounts
        .groupby(["analysis_month", col], dropna=False)
        .agg(
            accounts=("account_id", "nunique")
        )
        .reset_index()
    )

    temp["dimension"] = col
    temp = temp.rename(columns={col: "segment"})

    # Total accounts within each month and dimension
    monthly_totals = (
        temp
        .groupby("analysis_month")["accounts"]
        .transform("sum")
    )

    temp["share_pct"] = (
        temp["accounts"] / monthly_totals * 100
    )

    account_mix_results.append(temp)

# Safely combine results
if account_mix_results:

    portfolio_mix_df = pd.concat(
        account_mix_results,
        ignore_index=True
    )

    portfolio_mix_df = portfolio_mix_df[
        [
            "analysis_month",
            "dimension",
            "segment",
            "accounts",
            "share_pct"
        ]
    ]

    print(
        f"Portfolio mix records created: {len(portfolio_mix_df):,}"
    )

    display(portfolio_mix_df.head(50))

else:

    print(
        "No suitable low-cardinality categorical portfolio "
        "dimensions were found in accounts.csv."
    )

Portfolio dimensions found:
[]
No suitable low-cardinality categorical portfolio dimensions were found in accounts.csv.


In [58]:
#ACCOUNT POPULATION BY MONTH
accounts["opened_at_dt"] = pd.to_datetime(
    accounts["opened_at"],
    errors="coerce"
)

account_population = (
    accounts[
        accounts["opened_at_dt"].notna()
    ]
    .assign(
        month=lambda x: x["opened_at_dt"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        accounts=("account_id", "nunique"),
        total_outstanding=("outstanding_amount", "sum")
    )
    .reset_index()
)

account_population

,month,accounts,total_outstanding
0,2024-01,1311,4.527532e+08
1,2024-02,1299,4.553489e+08
2,2024-03,1404,4.982455e+08
3,2024-04,1263,4.277193e+08
4,2024-05,1287,4.442949e+08
5,2024-06,1219,4.267581e+08
6,2024-07,1324,4.652109e+08
7,2024-08,1358,4.840826e+08
8,2024-09,1326,4.722349e+08
9,2024-10,1362,4.720004e+08


In [59]:
#ACCOUNT STATUS DISTRIBUTION
status_distribution = (
    accounts["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="account_count")
)

status_distribution["share_pct"] = (
    status_distribution["account_count"]
    / len(accounts)
    * 100
)

status_distribution

,status,account_count,share_pct
0,ACTIVE,7539,25.130000
1,CLOSED,7496,24.986667
2,PAID,7486,24.953333
3,WRITEOFF,7479,24.930000


In [60]:
# ACCOUNT STATUS HISTORY COVERAGE

status_history = datasets["account_status_history"].copy()

status_history["event_at_dt"] = pd.to_datetime(
    status_history["event_at"],
    errors="coerce"
)

status_history["recorded_at_dt"] = pd.to_datetime(
    status_history["recorded_at"],
    errors="coerce"
)

print("Status history rows:", len(status_history))
print(
    "Unique accounts with status history:",
    status_history["account_id"].nunique()
)

print("\nStatus values:")
display(
    status_history["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="records")
)


Status history rows: 60000
Unique accounts with status history: 25999

Status values:


,status,records
0,PAID,8650
1,CLOSED,8614
2,DELINQUENT,8612
3,NPA,8612
4,WRITEOFF,8583
5,ACTIVE,8518
6,PTP,8411


In [61]:
#ACCOUNT STATUS TRANSITIONS
status_transitions = (
    status_history
    .sort_values(["account_id", "event_at_dt", "recorded_at_dt"])
    .groupby("account_id")
    .agg(
        status_events=("history_id", "count"),
        first_status=("status", "first"),
        last_status=("status", "last"),
        first_event=("event_at_dt", "min"),
        last_event=("event_at_dt", "max")
    )
    .reset_index()
)

print("Accounts represented in status history:", len(status_transitions))

display(
    status_transitions.head(20)
)

Accounts represented in status history: 25999


,account_id,status_events,first_status,last_status,first_event,last_event
0,ACC0000001,1,DELINQUENT,DELINQUENT,2026-02-05 18:41:49,2026-02-05 18:41:49
1,ACC0000002,3,PTP,DELINQUENT,2026-01-16 13:58:20,2026-05-15 13:04:13
2,ACC0000003,5,PAID,NPA,2026-01-31 11:25:03,2026-07-23 20:28:51
3,ACC0000004,1,ACTIVE,ACTIVE,2026-06-17 17:58:14,2026-06-17 17:58:14
4,ACC0000005,3,DELINQUENT,CLOSED,2026-01-24 18:59:39,2026-07-05 01:28:01
5,ACC0000006,4,PAID,PTP,2026-02-12 05:45:10,2026-06-18 13:32:39
6,ACC0000007,3,WRITEOFF,DELINQUENT,2026-01-26 19:56:26,2026-08-04 04:17:44
7,ACC0000008,3,ACTIVE,CLOSED,2026-02-06 23:27:14,2026-06-06 07:54:42
8,ACC0000009,2,NPA,PTP,2026-03-04 06:49:24,2026-03-11 04:27:34
9,ACC0000010,2,PTP,ACTIVE,2026-05-11 17:07:21,2026-06-26 16:53:04


In [62]:
#MULTIPLE STATUS CHANGE ANALYSIS
status_change_summary = pd.DataFrame({
    "metric": [
        "Total accounts",
        "Accounts with status history",
        "Accounts with multiple status events",
        "Accounts with 3+ status events"
    ],
    "count": [
        accounts["account_id"].nunique(),
        status_transitions["account_id"].nunique(),
        (status_transitions["status_events"] > 1).sum(),
        (status_transitions["status_events"] >= 3).sum()
    ]
})

status_change_summary

,metric,count
0,Total accounts,30000
1,Accounts with status history,25999
2,Accounts with multiple status events,17821
3,Accounts with 3+ status events,9664


In [63]:
#ACCOUNT VS PAYMENT POPULATION


successful_payment_accounts = set(
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .astype(str)
)

all_account_ids = set(
    accounts["account_id"]
    .dropna()
    .astype(str)
)

accounts_with_success = len(
    successful_payment_accounts & all_account_ids
)

accounts_without_success = len(
    all_account_ids - successful_payment_accounts
)

denominator_summary = pd.DataFrame({
    "population": [
        "All accounts",
        "Accounts with successful payment",
        "Accounts without successful payment"
    ],
    "accounts": [
        len(all_account_ids),
        accounts_with_success,
        accounts_without_success
    ]
})

denominator_summary["share_pct"] = (
    denominator_summary["accounts"]
    / len(all_account_ids)
    * 100
)

denominator_summary


,population,accounts,share_pct
0,All accounts,30000,100.00
1,Accounts with successful payment,13284,44.28
2,Accounts without successful payment,16716,55.72


In [64]:
# SCHEMA VERSION DISTRIBUTION
print("Schema versions in accounts:")

schema_versions = (
    accounts["schema_version"]
    .value_counts(dropna=False)
    .rename_axis("schema_version")
    .reset_index(name="account_count")
)

schema_versions["share_pct"] = (
    schema_versions["account_count"]
    / len(accounts)
    * 100
)

schema_versions

Schema versions in accounts:


,schema_version,account_count,share_pct
0,v1,10152,33.84
1,v2,10026,33.42
2,v3,9822,32.74


In [65]:
#SCHEMA VERSION OVER TIME

schema_over_time = (
    accounts[
        accounts["opened_at_dt"].notna()
    ]
    .assign(
        month=lambda x: x["opened_at_dt"].dt.to_period("M")
    )
    .groupby(["month", "schema_version"], dropna=False)
    .agg(
        accounts=("account_id", "nunique")
    )
    .reset_index()
)

schema_over_time["month_total"] = (
    schema_over_time
    .groupby("month")["accounts"]
    .transform("sum")
)

schema_over_time["share_pct"] = (
    schema_over_time["accounts"]
    / schema_over_time["month_total"]
    * 100
)

schema_over_time

,month,schema_version,accounts,month_total,share_pct
0,2024-01,v1,460,1311,35.087719
1,2024-01,v2,443,1311,33.790999
2,2024-01,v3,408,1311,31.121281
3,2024-02,v1,423,1299,32.563510
4,2024-02,v2,450,1299,34.642032
...,...,...,...,...,...
64,2025-10,v2,447,1325,33.735849
65,2025-10,v3,429,1325,32.377358
66,2025-11,v1,406,1254,32.376396
67,2025-11,v2,403,1254,32.137161


In [66]:
#CURRENT STATUS VS LATEST HISTORICAL STATUS



latest_status = (
    status_history[
        status_history["event_at_dt"].notna()
    ]
    .sort_values(
        ["account_id", "event_at_dt", "recorded_at_dt"]
    )
    .groupby("account_id")
    .tail(1)
    [["account_id", "status"]]
    .rename(columns={"status": "latest_historical_status"})
)

status_comparison = accounts[
    ["account_id", "status"]
].merge(
    latest_status,
    on="account_id",
    how="left"
)

status_comparison["status_matches"] = (
    status_comparison["status"]
    == status_comparison["latest_historical_status"]
)

status_comparison_summary = (
    status_comparison["status_matches"]
    .value_counts(dropna=False)
    .rename_axis("status_matches")
    .reset_index(name="accounts")
)

status_comparison_summary

,status_matches,accounts
0,False,26296
1,True,3704


In [67]:
#CALL DISPOSITION DISTRIBUTION
print("Call disposition columns:")
print(call_dispositions.columns.tolist())

print("\nUnique value counts:")

for col in call_dispositions.columns:
    if call_dispositions[col].dtype == "object":
        print(
            f"\n{col}: "
            f"{call_dispositions[col].nunique(dropna=True)} unique values"
        )

Call disposition columns:
['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version']

Unique value counts:


In [68]:
#IDENTIFY DISPOSITION FIELD
disposition_candidates = [
    col for col in call_dispositions.columns
    if any(keyword in col.lower() for keyword in [
        "disposition",
        "code",
        "status",
        "outcome"
    ])
]

print("Potential disposition fields:")
print(disposition_candidates)

Potential disposition fields:
['disposition_id', 'disposition_code', 'disposition_version']
